Imports & data loading (don't change!)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import loguniform, randint, uniform

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge, SGDRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
bike = fetch_openml(name='Bike_Sharing_Demand', as_frame=True, parser='auto')
df = bike.frame.copy()
print('Shape:', df.shape)
df.head()

Clean features (don't change!)

In [ ]:
LEAKY = ['casual', 'registered', 'datetime']
to_drop = [c for c in LEAKY if c in df.columns]
df = df.drop(columns=to_drop)
print('Dropped:', to_drop)
df.head()

Prepare features (don't change!)

In [ ]:
TARGET = 'count'
y = df[TARGET].astype(float).values
X = df.drop(columns=[TARGET])


cat_candidates = ['season', 'holiday', 'workingday', 'weather', 'weekday', 'month', 'year', 'hour']
cat_cols = [c for c in cat_candidates if c in X.columns]
print('Categorical cols:', cat_cols)

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X = X.astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print('X_train:', X_train.shape, '  X_test:', X_test.shape)

Visualize the data

In [ ]:
# I start with a small EDA step  
# before training regression models, 
# we need to understand the target distribution and the strongest numeric relations
plt.figure(figsize=(8, 5))
sns.histplot(y, bins=40, kde=True)
plt.title('Distribution of Bike Demand Count')
plt.xlabel('count')
plt.ylabel('frequency')
plt.show()

# if the original hour column exists: check the average demand per hour
if 'hour' in df.columns:
    plt.figure(figsize=(10, 5))
    hourly_demand = df.groupby('hour')[TARGET].mean()
    sns.lineplot(x=hourly_demand.index, y=hourly_demand.values, marker='o')
    plt.title('Average Bike Demand by Hour')
    plt.xlabel('hour')
    plt.ylabel('average count')
    plt.show()

# check correlations between numeric features and the target
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 7))
sns.heatmap(numeric_df.corr(), cmap='coolwarm', center=0)
plt.title('Correlation Heatmap of Numeric Features')
plt.show()

Train Random Forest

In [ ]:
results = [] # all model results

""" Evaluate one fitted RandomizedSearchCV object and store the test metrics:
calculate all metrics on the test set, 
print RMSE so RMSE is the direct interpretation of the CV score.
"""
def evaluate_and_store(model_name, search_object, X_test_data, y_test_data):
    predictions = search_object.predict(X_test_data)

    # bike counts cannot be negative
    predictions = np.maximum(predictions, 0)

    mse = mean_squared_error(y_test_data, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_data, predictions)
    r2 = r2_score(y_test_data, predictions)

    results.append({
        'Model': model_name,
        'Best CV RMSE': -search_object.best_score_,
        'Test MSE': mse,
        'Test RMSE': rmse,
        'Test MAE': mae,
        'Test R2': r2,
        'Best Params': search_object.best_params_
    })

    print(f'===== {model_name} =====')
    print('Best parameters:')
    print(search_object.best_params_)
    print(f'Best CV RMSE: {-search_object.best_score_:.4f}')
    print(f'Test MSE: {mse:.4f}')
    print(f'Test RMSE: {rmse:.4f}')
    print(f'Test MAE: {mae:.4f}')
    print(f'Test R²: {r2:.4f}')

# random forest
rf_model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)

# sample combinations with RandomizedSearchCV to create many parameter options
rf_param_distributions = {
    'n_estimators': randint(100, 500),
    'max_depth': [None, 5, 10, 15, 20, 30],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', 0.5, 0.7, 1.0],
    'bootstrap': [True, False]
}

rf_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_param_distributions,
    n_iter=35,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)
evaluate_and_store('Random Forest', rf_search, X_test, y_test)

Train Gradient Boosting

In [ ]:
# gradient boosting
gb_model = GradientBoostingRegressor(random_state=RANDOM_STATE)

gb_param_distributions = {
    'n_estimators': randint(100, 600),
    'learning_rate': loguniform(0.01, 0.2),
    'max_depth': randint(2, 6),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'subsample': uniform(0.6, 0.4),
    'max_features': ['sqrt', 'log2', None]
}

gb_search = RandomizedSearchCV(
    estimator=gb_model,
    param_distributions=gb_param_distributions,
    n_iter=35,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

gb_search.fit(X_train, y_train)
evaluate_and_store('Gradient Boosting', gb_search, X_test, y_test)

Train Ridge regression

In [ ]:
# since ridge depends on feature scale 
# I used StandardScaler inside a Pipeline
# because scaling must be learned only from the training folds in CV
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(random_state=RANDOM_STATE))
])

ridge_param_distributions = {
    'ridge__alpha': loguniform(1e-3, 1e3),
    'ridge__fit_intercept': [True, False],
    'ridge__solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sag', 'saga']
}

ridge_search = RandomizedSearchCV(
    estimator=ridge_pipeline,
    param_distributions=ridge_param_distributions,
    n_iter=35,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

ridge_search.fit(X_train, y_train)
evaluate_and_store('Ridge Regression', ridge_search, X_test, y_test)

Train SGDRegressor

In [ ]:
# SGDRegressor
sgd_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('sgd', SGDRegressor(random_state=RANDOM_STATE, max_iter=3000, tol=1e-3))
])

sgd_param_distributions = {
    'sgd__loss': ['squared_error', 'huber', 'epsilon_insensitive'],
    'sgd__penalty': ['l2', 'l1', 'elasticnet'],
    'sgd__alpha': loguniform(1e-6, 1e-1),
    'sgd__l1_ratio': uniform(0.05, 0.90),
    'sgd__learning_rate': ['constant', 'invscaling', 'adaptive'],
    'sgd__eta0': loguniform(1e-4, 1e-1),
    'sgd__average': [True, False]
}

sgd_search = RandomizedSearchCV(
    estimator=sgd_pipeline,
    param_distributions=sgd_param_distributions,
    n_iter=35,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

sgd_search.fit(X_train, y_train)
evaluate_and_store('SGDRegressor', sgd_search, X_test, y_test)

Add Feature engineering using PolynomialFeatures of degree=2 to create THE NEW DATASET

In [ ]:
# Now we create a new dataset using PolynomialFeatures with degree=2
# This adds squared terms and pairwise feature interactions
poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

print('Original training shape:', X_train.shape)
print('Polynomial training shape:', X_train_poly.shape)
print('Original test shape:', X_test.shape)
print('Polynomial test shape:', X_test_poly.shape)

# keep the feature names only for readability/debugging
# the models can work directly with the transformed numpy arrays
poly_feature_names = poly.get_feature_names_out(X_train.columns)
print('Number of polynomial features:', len(poly_feature_names))

Retrain Ridge regression (THE NEW DATASET!!!)




In [ ]:
# retrain Ridge on the polynomial dataset
poly_ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(random_state=RANDOM_STATE))
])

poly_ridge_param_distributions = {
    'ridge__alpha': loguniform(1e-3, 1e4),
    'ridge__fit_intercept': [True, False],
    'ridge__solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sag', 'saga']
}

poly_ridge_search = RandomizedSearchCV(
    estimator=poly_ridge_pipeline,
    param_distributions=poly_ridge_param_distributions,
    n_iter=35,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

poly_ridge_search.fit(X_train_poly, y_train)
evaluate_and_store('Ridge Regression + PolynomialFeatures', poly_ridge_search, X_test_poly, y_test)

Retrain SGDRegressor regression (THE NEW DATASET!!!)




In [ ]:
# retrain SGDRegressor on the polynomial dataset
# regularization are more important here
poly_sgd_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('sgd', SGDRegressor(random_state=RANDOM_STATE, max_iter=4000, tol=1e-3))
])

poly_sgd_param_distributions = {
    'sgd__loss': ['squared_error', 'huber', 'epsilon_insensitive'],
    'sgd__penalty': ['l2', 'l1', 'elasticnet'],
    'sgd__alpha': loguniform(1e-6, 1e-1),
    'sgd__l1_ratio': uniform(0.05, 0.90),
    'sgd__learning_rate': ['constant', 'invscaling', 'adaptive'],
    'sgd__eta0': loguniform(1e-4, 1e-1),
    'sgd__average': [True, False]
}

poly_sgd_search = RandomizedSearchCV(
    estimator=poly_sgd_pipeline,
    param_distributions=poly_sgd_param_distributions,
    n_iter=35,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

poly_sgd_search.fit(X_train_poly, y_train)
evaluate_and_store('SGDRegressor + PolynomialFeatures', poly_sgd_search, X_test_poly, y_test)

Train Neural Network regression ( ORIGINAL DATASET!!!)


In [ ]:
# neural networks are sensitive to scale, so I use StandardScaler inside the Pipeline
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(
        random_state=RANDOM_STATE,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.15
    ))
])

mlp_param_distributions = {
    'mlp__hidden_layer_sizes': [(32,), (64,), (128,), (64, 32), (128, 64), (100, 50)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__solver': ['adam'],
    'mlp__alpha': loguniform(1e-5, 1e-1),
    'mlp__learning_rate_init': loguniform(1e-4, 1e-2),
    'mlp__batch_size': [32, 64, 128, 256]
}

mlp_search = RandomizedSearchCV(
    estimator=mlp_pipeline,
    param_distributions=mlp_param_distributions,
    n_iter=25,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

mlp_search.fit(X_train, y_train)
evaluate_and_store('MLPRegressor', mlp_search, X_test, y_test)

Perform the final comparison between the results

In [ ]:
# final comparison of all trained models:
# lower MSE / RMSE / MAE is better, and higher R^2 is better
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Test RMSE').reset_index(drop=True)
results_df.index = results_df.index + 1

# display the full comparison table first:
display(results_df[['Model', 'Best CV RMSE', 'Test MSE', 'Test RMSE', 'Test MAE', 'Test R2']])

best_model_name = results_df.iloc[0]['Model']
print(f'The best model according to Test RMSE is: {best_model_name}')
print('Best model parameters:')
print(results_df.iloc[0]['Best Params'])

# visualize ranking by Test RMSE:
plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x='Test RMSE', y='Model')
plt.title('Model Comparison by Test RMSE')
plt.xlabel('Test RMSE')
plt.ylabel('Model')
plt.show()

# My final conclusion is based mainly on Test RMSE because the hyperparameter search
# was optimized using neg_root_mean_squared_error. I still keep MSE, MAE, and R^2 in
# the table because the assignment explicitly asks to evaluate every model by them.